In [5]:
import numpy as np

In [28]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784')
X = mnist.data.values.astype('float32')
# huge number will make the training explode for weights and biases 
# 255 specificly since it is the actual max pixel value 8 bit
# Dividing itgives exact 0-1 range
X = X/255.0
y = mnist.target.values.astype(int)

In [29]:
class Layers:
    def __init__(self, n_inputs, n_neurons):
        self.weight = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.output = np.dot(inputs, self.weight) + self.biases

In [30]:
l1 = Layers(784,128)
l1.forward(X)
l1.output[::5]

array([[-0.0118642 ,  0.06798438, -0.08574459, ..., -0.00990891,
        -0.07647943, -0.03884787],
       [-0.07413141, -0.07728715,  0.04439121, ..., -0.0763338 ,
        -0.07340089,  0.06870976],
       [ 0.03199011,  0.0646336 ,  0.00497617, ..., -0.03377019,
         0.06256934,  0.14764682],
       ...,
       [ 0.01752511,  0.04652612, -0.00154561, ...,  0.06440108,
        -0.00173889,  0.07685706],
       [ 0.1040996 , -0.04592688,  0.01810059, ...,  0.02690487,
         0.068451  ,  0.19386963],
       [ 0.03082497,  0.00407635,  0.11954485, ..., -0.01053357,
         0.08904366,  0.18904114]], shape=(14000, 128))

In [31]:
class ReLU:
    def forward(self, input):
        self.output = np.maximum(0, input)

In [33]:
relu = ReLU()
relu.forward(l1.output)
l1.output.shape

(70000, 128)

In [37]:
l2 = Layers(128,10)
l2.forward(relu.output)
l2.output[::10]

array([[-0.00600733, -0.02317867, -0.00919138, ..., -0.00250102,
        -0.00677223, -0.00054219],
       [-0.0013328 , -0.01658359,  0.00237454, ..., -0.00328665,
        -0.00010683,  0.00171152],
       [-0.00238463, -0.01836932, -0.00935408, ...,  0.00206083,
        -0.00676594, -0.00694411],
       ...,
       [-0.00329502, -0.00465624, -0.00102248, ...,  0.00278219,
         0.00148439, -0.00122706],
       [-0.01217165, -0.02453475, -0.00741272, ...,  0.01368934,
         0.00076779, -0.00800748],
       [-0.00632976, -0.01438416,  0.00320699, ..., -0.00240016,
        -0.00887157, -0.00623578]], shape=(7000, 10))

In [38]:
class SoftMax:
    def forward(self, input):
        exp_values = np.exp(input-np.max(input, axis=1, keepdims=True))
        probablities = exp_values/ np.sum(exp_values, axis=1, keepdims=True)
        self.output = probablities

In [39]:
softmax = SoftMax()
softmax.forward(l2.output)

In [40]:
softmax.output

array([[0.09990607, 0.09820519, 0.09958847, ..., 0.10025699, 0.09982968,
        0.10045357],
       [0.09930729, 0.09933665, 0.10088668, ..., 0.10018273, 0.10032535,
        0.0996308 ],
       [0.10015563, 0.09961151, 0.09944186, ..., 0.10028524, 0.09934534,
        0.10041105],
       ...,
       [0.09998669, 0.09952221, 0.10001183, ..., 0.10004783, 0.09962521,
        0.10075105],
       [0.09940057, 0.09924613, 0.09960167, ..., 0.10041376, 0.10038538,
        0.10045829],
       [0.09960564, 0.09942987, 0.09963642, ..., 0.1005443 , 0.0998321 ,
        0.09954708]], shape=(70000, 10))

In [47]:
class Loss:
    def calculate(self, output, y):
        sample_loss = self.forward(output, y)
        data_loss = np.mean(sample_loss)
        return data_loss
class CrossEntrophyLoss(Loss):
    def forward(self, y_pred, y_true):
        sample = len(y_pred)
        y_clipped = np.clip(y_pred, 1e-7, 1-1e-7)
        if len(y_true.shape) == 1:
            correct_confidences = y_clipped[
                range(sample),
                y_true
            ]
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(y_clipped * y_true, axis=1)
        
        neg_log_likelihoods = -np.log(correct_confidences)
        return neg_log_likelihoods

In [51]:
entrophy = CrossEntrophyLoss()
loss = entrophy.forward(softmax.output, y)
print(entrophy.calculate(softmax.output, y))

2.3033125884401513


In [52]:
predictions = np.argmax(softmax.output, axis=1)
accuracy = np.mean(predictions == y)
print('acc:', accuracy)

acc: 0.11432857142857143
